# DAPA: load sample data

Builds the gold tables behind the DAPA dashboards in the default lakehouse.

**All data produced here is synthetic sample data.** Tract IDs, banks and attendees are invented.
Replace each table with its real pipeline (ACS, FDIC, Inclusive Growth Score, event capture, bank outcomes) as those land.

| Table | Feeds | Real source later |
|---|---|---|
| `dim_tract` | Opportunity map, control comparison | ACS 5-year, FDIC survey, Inclusive Growth Score, business listings |
| `dim_bank`, `dim_branch` | Branch overlay, bank fit | FDIC Summary of Deposits |
| `dim_event`, `fact_attendee` | Bank pilot dashboard | Event registration, attendance, surveys, bank outcomes (bank tenant only) |
| `scenario_ltv` | Scenario view | Azure ML lifetime value model (labeled ranges) |
| `agg_playbook` | Mastercard playbook | De-identified, small-group-suppressed aggregate of the above |

Run it in Fabric with the DAPA lakehouse attached. Run it locally with plain Python and it writes CSVs to `sample-output/` instead.

In [ ]:
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42  # fixed so every run produces the same sample
MIN_GROUP_SIZE = 10  # N2: Mastercard-view groups with fewer attendees than this are dropped
TENANT_BANK_ID = "BANK-01"  # the bank whose Azure tenant this workspace lives in
PILOT_START = date(2026, 7, 1)
rng = np.random.default_rng(SEED)


def pct_rank(s: pd.Series, ascending: bool = True) -> pd.Series:
    """Score in (0, 1] by percentile rank, so no tract scores exactly zero."""
    return s.rank(pct=True, ascending=ascending)

## Tracts and opportunity score (F1, F2)

Score = need x readiness to adopt x bank fit. Targeting uses place and need only (N5):
no race, ethnicity or national-origin fields are used. Language is kept for outreach planning, not scoring.

In [ ]:
N_TRACTS = 16
ids = np.arange(1, N_TRACTS + 1)
dim_tract = pd.DataFrame({
    "tract_id": [f"SAMPLE-{i:03d}" for i in ids],
    "tract_name": [f"Sample Tract {i:03d}" for i in ids],
    "county": np.where(ids <= N_TRACTS // 2, "Sample County North", "Sample County South"),
    "state": "IL",
    "latitude": (41.85 + rng.uniform(-0.12, 0.12, N_TRACTS)).round(5),
    "longitude": (-87.68 + rng.uniform(-0.12, 0.12, N_TRACTS)).round(5),
    "adult_population": rng.integers(1500, 6000, N_TRACTS),
    "unbanked_rate": rng.uniform(0.02, 0.18, N_TRACTS).round(3),
    "underbanked_rate": rng.uniform(0.08, 0.30, N_TRACTS).round(3),
    "low_income_share": rng.uniform(0.10, 0.60, N_TRACTS).round(3),
    "alt_financial_outlets": rng.integers(0, 12, N_TRACTS),
    "smartphone_rate": rng.uniform(0.70, 0.95, N_TRACTS).round(3),
    "broadband_rate": rng.uniform(0.55, 0.92, N_TRACTS).round(3),
    "small_business_per_1k": rng.uniform(5, 40, N_TRACTS).round(1),
    "inclusive_growth_score": rng.uniform(20, 80, N_TRACTS).round(1),
    "top_non_english_language": rng.choice(["Spanish", "Polish", "Arabic", "Tagalog", "Chinese"], N_TRACTS),
    "limited_english_rate": rng.uniform(0.02, 0.35, N_TRACTS).round(3),
})

## Banks and branches (F1, F3)

Branch locations are public (FDIC), so other banks' branches can appear in the overlay.
Event and outcome data below exist only for the tenant bank.

In [ ]:
dim_bank = pd.DataFrame({
    "bank_id": ["BANK-01", "BANK-02", "BANK-03"],
    "bank_name": ["Sample Community Bank", "Sample Credit Union", "Sample National Bank"],
    "bank_type": ["Community bank under $10B", "Credit union", "National bank"],
    "total_assets_usd_bn": [2.4, 0.8, 450.0],
    "is_tenant_bank": [True, False, False],
})

N_BRANCHES = 10
branch_tracts = dim_tract.sample(N_BRANCHES, replace=True, random_state=SEED)
dim_branch = pd.DataFrame({
    "branch_id": [f"BR-{i:03d}" for i in range(1, N_BRANCHES + 1)],
    "bank_id": rng.choice(dim_bank["bank_id"], N_BRANCHES, p=[0.4, 0.3, 0.3]),
    "tract_id": branch_tracts["tract_id"].to_numpy(),
    "latitude": (branch_tracts["latitude"].to_numpy() + rng.uniform(-0.005, 0.005, N_BRANCHES)).round(5),
    "longitude": (branch_tracts["longitude"].to_numpy() + rng.uniform(-0.005, 0.005, N_BRANCHES)).round(5),
    "deposits_usd_mm": rng.uniform(20, 400, N_BRANCHES).round(1),
})
dim_branch.loc[0, "bank_id"] = TENANT_BANK_ID  # the tenant bank always has at least one branch

# Bank fit: closer to a tenant-bank branch scores higher.
tenant = dim_branch[dim_branch["bank_id"] == TENANT_BANK_ID]
dist = np.sqrt(
    (dim_tract["latitude"].to_numpy()[:, None] - tenant["latitude"].to_numpy()) ** 2
    + (dim_tract["longitude"].to_numpy()[:, None] - tenant["longitude"].to_numpy()) ** 2
).min(axis=1)

outlets_per_1k = dim_tract["alt_financial_outlets"] / dim_tract["adult_population"] * 1000
dim_tract["need_score"] = pd.concat([
    pct_rank(dim_tract["unbanked_rate"]),
    pct_rank(dim_tract["underbanked_rate"]),
    pct_rank(dim_tract["low_income_share"]),
    pct_rank(outlets_per_1k),
    pct_rank(dim_tract["inclusive_growth_score"], ascending=False),
], axis=1).mean(axis=1).round(3)
dim_tract["readiness_score"] = pd.concat([
    pct_rank(dim_tract["smartphone_rate"]),
    pct_rank(dim_tract["broadband_rate"]),
    pct_rank(dim_tract["small_business_per_1k"]),
], axis=1).mean(axis=1).round(3)
dim_tract["bank_fit_score"] = pct_rank(pd.Series(dist), ascending=False).round(3).to_numpy()
dim_tract["opportunity_score"] = (
    dim_tract["need_score"] * dim_tract["readiness_score"] * dim_tract["bank_fit_score"]
).round(3)

# Top four tracts get the two pilot arms; the next two, closest in score, are the matched controls.
order = dim_tract["opportunity_score"].rank(ascending=False, method="first").astype(int)
dim_tract["pilot_arm"] = np.select(
    [order.isin([1, 3]), order.isin([2, 4]), order.isin([5, 6])],
    ["Pilot A", "Pilot B", "Control"],
    default="Not selected",
)

## Events and attendees (F7, F8)

One row per registration, keyed by an event referral code (no names or contact details).
Outcomes are matched only for attendees who opted in (N4); everyone else has empty outcome fields.

In [ ]:
PRODUCT_FOCUS = {
    "Pilot A": "Credit builder + multilingual education",
    "Pilot B": "Remittance account + spend-and-get",
}
EVENT_TYPES = ["Community workshop", "Cultural festival booth", "Faith-based partnership session", "Small business clinic"]
APPROACHES = ["Trusted messenger", "Bank-led", "Peer referral"]
HOSTS = ["Community organization", "Church", "Employer", "Bank On coalition"]
CHANNELS = ["Walk-in", "Community partner", "Flyer QR code", "Text message"]

events, attendees = [], []
pilot_tracts = dim_tract[dim_tract["pilot_arm"].isin(PRODUCT_FOCUS)]
for _, t in pilot_tracts.iterrows():
    for _ in range(3):
        event_id = f"EVT-{len(events) + 1:03d}"
        approach = rng.choice(APPROACHES)
        events.append({
            "event_id": event_id,
            "bank_id": TENANT_BANK_ID,
            "tract_id": t["tract_id"],
            "event_date": PILOT_START + timedelta(days=int(rng.integers(0, 60))),
            "event_type": rng.choice(EVENT_TYPES),
            "activities": rng.choice(["Account opening desk", "Budgeting game", "Fraud-safety demo", "Remittance cost comparison"]),
            "approach": approach,
            "host_org_type": rng.choice(HOSTS),
            "language": rng.choice(["English", t["top_non_english_language"]]),
            "product_focus": PRODUCT_FOCUS[t["pilot_arm"]],
            "cost_usd": float(rng.integers(1500, 6000)),
        })
        trusted = approach == "Trusted messenger"
        for n in range(int(rng.integers(25, 70))):
            attended = bool(rng.random() < (0.75 if trusted else 0.6))
            consented = bool(rng.random() < 0.85)
            before = int(rng.integers(1, 4))
            surveyed = attended and consented
            matched = attended and consented
            opened = matched and bool(rng.random() < (0.35 if trusted else 0.25))
            funded = opened and bool(rng.random() < 0.75)
            active = funded and bool(rng.random() < 0.7)
            attendees.append({
                "referral_code": f"{event_id}-R{n + 1:03d}",
                "event_id": event_id,
                "channel": rng.choice(CHANNELS),
                "consented": consented,
                "attended": attended,
                "trust_before": before if surveyed else None,
                "trust_after": min(5, before + int(rng.integers(0, 3))) if surveyed else None,
                "account_opened": opened if matched else None,
                "funded_30d": funded if matched else None,
                "active_90d": active if matched else None,
                "products_held": int(rng.integers(1, 4)) if active else (0 if matched else None),
            })

dim_event = pd.DataFrame(events)
fact_attendee = pd.DataFrame(attendees)

# Observed new active accounts per tract during the pilot window: organic baseline everywhere,
# plus event-driven accounts in pilot tracts. The control arm shows the baseline (matched control).
baseline = rng.poisson(2.0 * dim_tract["adult_population"] / 1000)
event_active = (
    fact_attendee[fact_attendee["active_90d"] == True]  # noqa: E712 (column holds None too)
    .merge(dim_event[["event_id", "tract_id"]], on="event_id")
    .groupby("tract_id").size()
)
dim_tract["observed_new_active_accounts"] = (
    baseline + dim_tract["tract_id"].map(event_active).fillna(0).astype(int).to_numpy()
)

## Scenario: lifetime value ranges (F11, N6)

These are **assumptions**, shown as Low / Base / High ranges and labeled as such in the report.
Replace with the Azure ML lifetime value model, then with pilot actuals.

In [ ]:
ASSUMPTIONS = {  # scenario: (annual revenue per active account in USD, yearly retention)
    "Low": (60, 0.70),
    "Base": (110, 0.80),
    "High": (180, 0.88),
}
scenario_ltv = pd.DataFrame([
    {
        "scenario": name,
        "year": year,
        "cumulative_ltv_usd": round(sum(revenue * retention ** (y - 1) for y in range(1, year + 1)), 2),
        "basis": "Assumption",
    }
    for name, (revenue, retention) in ASSUMPTIONS.items()
    for year in range(1, 6)
])

## Mastercard playbook aggregate (F10, N1, N2)

The only table the Mastercard view reads. No referral codes or tract-level rows: results roll up
to county, product focus, event type and approach, and any group under `MIN_GROUP_SIZE` attendees is dropped.

In [ ]:
joined = (
    fact_attendee
    .merge(dim_event, on="event_id")
    .merge(dim_tract[["tract_id", "county"]], on="tract_id")
    .merge(dim_bank[["bank_id", "bank_type"]], on="bank_id")
)
joined["trust_lift"] = joined["trust_after"] - joined["trust_before"]
joined["matched"] = joined["attended"] & joined["consented"]
keys = ["county", "product_focus", "event_type", "approach", "bank_type"]
agg_playbook = joined.groupby(keys).agg(
    registrations=("referral_code", "size"),
    attendees=("attended", "sum"),
    matched_attendees=("matched", "sum"),
    accounts_opened=("account_opened", lambda s: int((s == True).sum())),  # noqa: E712
    active_90d=("active_90d", lambda s: int((s == True).sum())),  # noqa: E712
    trust_lift_sum=("trust_lift", "sum"),
    survey_responses=("trust_lift", "count"),
).reset_index()
event_stats = joined.drop_duplicates("event_id").groupby(keys).agg(
    events=("event_id", "size"), cost_usd=("cost_usd", "sum"),
).reset_index()
agg_playbook = (
    agg_playbook.merge(event_stats, on=keys)
    .rename(columns={"county": "area", "bank_type": "contributor_bank_type"})
)
agg_playbook = agg_playbook[agg_playbook["attendees"] >= MIN_GROUP_SIZE].reset_index(drop=True)
for col in ["registrations", "attendees", "matched_attendees", "events", "survey_responses"]:
    agg_playbook[col] = agg_playbook[col].astype(int)
agg_playbook["trust_lift_sum"] = agg_playbook["trust_lift_sum"].astype(float)

## Write tables

In Fabric: overwrite Delta tables in the attached lakehouse. Locally: write CSVs to `sample-output/`.

In [ ]:
TABLES = {
    "dim_tract": dim_tract,
    "dim_bank": dim_bank,
    "dim_branch": dim_branch,
    "dim_event": dim_event,
    "fact_attendee": fact_attendee,
    "scenario_ltv": scenario_ltv,
    "agg_playbook": agg_playbook,
}


def to_records(df: pd.DataFrame) -> list:
    """Native Python values with None for missing, so Spark infers nullable columns cleanly."""
    rows = df.astype(object).where(df.notna(), None).to_dict("records")
    return [{k: v.item() if isinstance(v, np.generic) else v for k, v in row.items()} for row in rows]


if "spark" in globals():
    for name, df in TABLES.items():
        (
            spark.createDataFrame(to_records(df))  # noqa: F821 (provided by the Fabric runtime)
            .select(*df.columns)
            .write.mode("overwrite").option("overwriteSchema", "true")
            .format("delta").saveAsTable(name)
        )
        print(f"wrote {name}: {len(df)} rows")
else:
    out = Path("sample-output")
    out.mkdir(exist_ok=True)
    for name, df in TABLES.items():
        df.to_csv(out / f"{name}.csv", index=False)
        print(f"wrote {out / name}.csv: {len(df)} rows")